In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [2]:
print(os.getcwd())

/Users/kimseungyeon1/Desktop/리스크


In [3]:
data=pd.read_pickle('/Users/kimseungyeon1/Desktop/리스크/Dataset of health insurance portfolio/Health_Insurance_final.pkl')
data

,lapse,exposure_time,premium,seniority_insured,seniority_policy,type_policy_dg,type_product,new_business,cost_claims_year,n_medical_services,...,IICIMUN_capped,help_requested,high_loss,relative_poverty,kr_age_risk,kr_premium_shock,kr_shock_amount,kr_economic_stress,kr_early_laps,kr_medical_desert
0,1,0.969749,2923.320000,24,24,I,S,No,8242.62,32,...,22.842273,1,2.819609,0.858586,초고령,1.1500,3361.818000,33.957758,0,3.659259e-05
1,2,1.000000,1930.110000,24,24,I,S,No,344.41,22,...,22.842273,1,0.178441,0.858586,초고령,1.1500,2219.626500,22.420470,0,3.659259e-05
2,2,1.000000,2020.000000,24,24,I,P,No,523.18,24,...,17.132586,1,0.259000,0.940000,초고령,1.1500,2323.000000,23.230000,0,3.891220e-07
3,2,1.000000,1843.250000,24,24,I,S,No,257.55,29,...,-1.000000,1,0.139726,-1.000000,초고령,1.1500,2119.737500,-1.000000,0,-1.000000e+00
4,2,1.000000,1316.030000,24,24,I,S,No,420.16,9,...,17.132586,1,0.319263,0.818182,장년,1.0500,1381.831500,13.957894,0,3.891220e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228705,2,1.000000,581.748774,0,0,I,S,Yes,0.00,0,...,0.097371,0,0.000000,0.205128,중년,1.0549,613.686782,15.735559,1,1.333333e-04
228706,3,1.000000,627.435000,0,0,I,S,Yes,0.00,0,...,6.898817,0,0.000000,0.528090,중년,1.0549,661.881181,7.436867,1,2.635498e-05
228707,2,1.000000,522.972000,0,0,I,S,Yes,0.00,0,...,1.538856,0,0.000000,0.470588,중년,1.0549,551.683163,6.490390,1,3.713333e-05
228708,2,1.000000,522.972000,0,0,I,S,Yes,0.00,0,...,1.538856,0,0.000000,0.470588,중년,1.0549,551.683163,6.490390,1,3.713333e-05


In [4]:
# lapse 값 변환: 1, 3 → 1 / 2 → 0
data['lapse'] = data['lapse'].map({1: 1, 3: 1, 2: 0})

In [5]:
import pandas as pd

# cat_cols가 정의되어 있고, 전체 데이터프레임 이름이 'data', 연도 컬럼이 'period'라고 가정합니다.
cat_cols = ['type_policy_dg', 'type_product', 'new_business', 'distribution_channel', 'gender', 'C_C', 'kr_early_laps']

print("🔍 [범주형 변수 연도별(2017, 2018, 2019) 카테고리 비교 분석]\n" + "="*60)

for col in cat_cols:
    # 각 연도별 고유값(범주)을 추출하여 집합(Set)으로 변환 (결측치는 제외)
    cats_2017 = set(data[data['period'] == 2017][col].dropna().unique())
    cats_2018 = set(data[data['period'] == 2018][col].dropna().unique())
    cats_2019 = set(data[data['period'] == 2019][col].dropna().unique())

    # 3개 연도의 범주가 완벽하게 일치하는지 확인
    if cats_2017 == cats_2018 == cats_2019:
        print(f"✅ [{col}] : 모든 연도의 범주가 완벽히 동일합니다.")
    else:
        print(f"⚠️ [{col}] : 연도별 범주에 차이가 있습니다!")
        
        # 모델 학습 관점에서 가장 중요한 분석: Train(17+18) vs Test(19) 비교
        train_cats = cats_2017.union(cats_2018) # 2017년과 2018년 범주의 합집합
        test_cats = cats_2019
        
        # 차집합을 이용해 원인 분석
        new_in_test = test_cats - train_cats
        missing_in_test = train_cats - test_cats
        
        if new_in_test:
            print(f"    🚨 위험! 2019년(Test)에 처음 등장한 미지의 범주: {new_in_test}")
        if missing_in_test:
            print(f"    ℹ️ 참고: 17/18년엔 있었으나 2019년(Test)엔 등장하지 않은 범주: {missing_in_test}")
            
    print("-" * 60)

🔍 [범주형 변수 연도별(2017, 2018, 2019) 카테고리 비교 분석]
✅ [type_policy_dg] : 모든 연도의 범주가 완벽히 동일합니다.
------------------------------------------------------------
✅ [type_product] : 모든 연도의 범주가 완벽히 동일합니다.
------------------------------------------------------------
✅ [new_business] : 모든 연도의 범주가 완벽히 동일합니다.
------------------------------------------------------------
✅ [distribution_channel] : 모든 연도의 범주가 완벽히 동일합니다.
------------------------------------------------------------
✅ [gender] : 모든 연도의 범주가 완벽히 동일합니다.
------------------------------------------------------------
✅ [C_C] : 모든 연도의 범주가 완벽히 동일합니다.
------------------------------------------------------------
✅ [kr_early_laps] : 모든 연도의 범주가 완벽히 동일합니다.
------------------------------------------------------------


In [6]:
import torch
target = 'lapse'

x_cols = [
    # 기본
    'premium', 'seniority_policy', 'type_policy_dg', 'type_product', 'new_business',
    'log_cost_claims_year', 'distribution_channel',
    # 나이 관련
    'age',
    # 성별 관련
    'gender',
    # 지역 관련
    'IICIMUN_capped', 'IICIPROV', 'C_C', 'C_H_num', 'C_GI', 'C_IE_T',
]
#파생변수
der_cols = [
    'missing_geo_cxt',       # 지역 결측 신호
    'high_loss',                  # 보험사 손해율
    'relative_poverty',        # 지역 내 상대적 빈곤
    'kr_premium_shock',   # 가격 인상 압박(비율 단위)
    'kr_economic_stress',  # 소득 대비 체감 부담
    # 'kr_retention_years',    # 장기 유지 혜택
    'kr_early_laps',             #신계약 위험 구간
    # 'kr_direct_channel',     # 가입 채널 영향
    'kr_medical_desert'     # 인프라 취약성
]

print(f"사용 변수 개수(기본) : {len(x_cols)}")
print(f"사용 변수 개수(파생) : {len(der_cols)}")
#기본변수+파생변수 (x_cols)
x_cols += der_cols
print(f"최종 변수 개수 : {len(x_cols)}개")

device = "cuda" if torch.cuda.is_available() else "cpu"

사용 변수 개수(기본) : 15
사용 변수 개수(파생) : 7
최종 변수 개수 : 22개


In [7]:
### Encoding
cat_cols = ['type_policy_dg', 'type_product', 'new_business', 'distribution_channel', 'gender', 'C_C', 'kr_early_laps']

In [8]:
#범주형 변수 인코딩 위해 모든 범주 type str로 변경 (C_C변수 포함)
data[cat_cols] = data[cat_cols].astype(str)

In [9]:
num_cols = [col for col in x_cols if col not in cat_cols]

In [10]:
#데이터셋 분리 (시계열 데이터임을 고려해 연도별 분리)
train = data[data['period'] == 2017]
val   = data[data['period'] == 2018]
test  = data[data['period'] == 2019]

X_train, y_train = train[x_cols], train[target]
X_val, y_val     = val[x_cols], val[target]
X_test, y_test   = test[x_cols], test[target]

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [12]:
#연도별 해지율 확인
data.groupby('period')['lapse'].mean()

period
2017    0.189977
2018    0.190435
2019    0.161167
Name: lapse, dtype: float64

- 전처리 (표준화, 인코딩)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),   #수치형 표준화 
        ('cat', OneHotEncoder(drop='first'), cat_cols)  #범주형 표준화 
    ])


In [14]:
print("데이터 전처리(스케일링 및 원핫인코딩)")
# Train 기준으로 학습(fit) 후 변환 -> 2017,2018같이 스케일링 할 경우 데이터 누수 가능성 있음
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

데이터 전처리(스케일링 및 원핫인코딩)


#### optuna

In [15]:
!pip install optuna

#### optuna에서 auc대신 pr-auc사용

In [ ]:
import optuna
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline # 파이프라인 임포트
import warnings
from sklearn.base import clone

def objective(trial):
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)
    weight_type = trial.suggest_categorical('weight_type', ['none', 'balanced', 'custom'])
    if weight_type == 'none':
        class_weight = None
    elif weight_type == 'balanced':
        class_weight = 'balanced'
    else:
        churn_weight = trial.suggest_float('churn_weight', 1.0, 8.0) 
        class_weight = {0: 1.0, 1: churn_weight}

    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

    model = LogisticRegression(
        C=C,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        penalty='elasticnet',
        solver='saga',   
        max_iter=3000,  
        tol=1e-4,   
        random_state=42,
    )


    optuna_pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', model)
    ])

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        
        optuna_pipeline.fit(X_train, y_train)  #파이프라인 사용 -> 스케일링, 인코딩 전 원본데이터 사용
        
        preds_proba = optuna_pipeline.predict_proba(X_val)[:, 1]

    prauc = average_precision_score(y_val, preds_proba)
    return prauc

study = optuna.create_study(
    direction='maximize',   #기준 점수 prauc극대화 
    sampler=optuna.samplers.TPESampler(seed=42)   #하이퍼파라미터 무작위로 찍는 것이 아니라, 과거 실험 바탕으로 성공확률 높은 영역 찾아가는 베이지안 최적화기반 알고리즘 사용
)
study.optimize(objective, n_trials=300)  #하이퍼파라미터 시도 조합 개수

print("\n" + "="*40)
print(f"최적 Val PR-AUC : {study.best_value:.4f}")
print(f"최적 파라미터 : {study.best_params}")
print("="*40)

[I 2026-05-18 14:43:12,686] A new study created in memory with name: no-name-cf156ead-2ef8-4eb4-8554-800e6f0b8394
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
[I 2026-05-18 14:43:15,632] Trial 0 finished with value: 0.32457531183978217 and parameters: {'C': 0.0074593432857265485, 'weight_type': 'none', 'l1_ratio': 0.15601864044243652}. Best is trial 0 with value: 0.32457531183978217.
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' in


최적 Val PR-AUC : 0.3410
최적 파라미터 : {'C': 0.002339068396169644, 'weight_type': 'none', 'l1_ratio': 0.9983179962329991}


- C범위 (1e-4,10) -> (1e-4, 1)

In [16]:
import optuna
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline # 파이프라인 임포트
import warnings
from sklearn.base import clone

def objective(trial):
    C = trial.suggest_float('C', 1e-4, 1, log=True)
    weight_type = trial.suggest_categorical('weight_type', ['none', 'balanced', 'custom'])
    if weight_type == 'none':
        class_weight = None
    elif weight_type == 'balanced':
        class_weight = 'balanced'
    else:
        churn_weight = trial.suggest_float('churn_weight', 1.0, 8.0) 
        class_weight = {0: 1.0, 1: churn_weight}

    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

    model = LogisticRegression(
        C=C,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        penalty='elasticnet',
        solver='saga',   
        max_iter=3000,  
        tol=1e-4,   
        random_state=42,
    )


    optuna_pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', model)
    ])

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        
        optuna_pipeline.fit(X_train, y_train)  #파이프라인 사용 -> 스케일링, 인코딩 전 원본데이터 사용
        
        preds_proba = optuna_pipeline.predict_proba(X_val)[:, 1]

    prauc = average_precision_score(y_val, preds_proba)
    return prauc

study = optuna.create_study(
    direction='maximize',   #기준 점수 prauc극대화 
    sampler=optuna.samplers.TPESampler(seed=42)   #하이퍼파라미터 무작위로 찍는 것이 아니라, 과거 실험 바탕으로 성공확률 높은 영역 찾아가는 베이지안 최적화기반 알고리즘 사용
)
study.optimize(objective, n_trials=300)  #하이퍼파라미터 시도 조합 개수

print("\n" + "="*40)
print(f"최적 Val PR-AUC : {study.best_value:.4f}")
print(f"최적 파라미터 : {study.best_params}")
print("="*40)

[I 2026-07-23 15:20:26,566] A new study created in memory with name: no-name-86472d5c-8904-42fa-aa8c-ef8eb40985f1
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
[I 2026-07-23 15:20:28,132] Trial 0 finished with value: 0.3275388705568347 and parameters: {'C': 0.003148911647956862, 'weight_type': 'none', 'l1_ratio': 0.15601864044243652}. Best is trial 0 with value: 0.3275388705568347.
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning,


최적 Val PR-AUC : 0.3409
최적 파라미터 : {'C': 0.002277236180985177, 'weight_type': 'none', 'l1_ratio': 0.9998171197302008}


- C 범위 (1e-4, 1) -> (1e-3, 1e-2)

In [17]:
import optuna
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline # 파이프라인 임포트
import warnings
from sklearn.base import clone

def objective(trial):
    C = trial.suggest_float('C', 1e-3, 1e-2, log=True)
    weight_type = trial.suggest_categorical('weight_type', ['none', 'balanced', 'custom'])
    if weight_type == 'none':
        class_weight = None
    elif weight_type == 'balanced':
        class_weight = 'balanced'
    else:
        churn_weight = trial.suggest_float('churn_weight', 1.0, 8.0) 
        class_weight = {0: 1.0, 1: churn_weight}

    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

    model = LogisticRegression(
        C=C,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        penalty='elasticnet',
        solver='saga',   
        max_iter=3000,  
        tol=1e-4,   
        random_state=42,
    )


    optuna_pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', model)
    ])

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        
        optuna_pipeline.fit(X_train, y_train)  #파이프라인 사용 -> 스케일링, 인코딩 전 원본데이터 사용
        
        preds_proba = optuna_pipeline.predict_proba(X_val)[:, 1]

    prauc = average_precision_score(y_val, preds_proba)
    return prauc

study = optuna.create_study(
    direction='maximize',   #기준 점수 prauc극대화 
    sampler=optuna.samplers.TPESampler(seed=42)   #하이퍼파라미터 무작위로 찍는 것이 아니라, 과거 실험 바탕으로 성공확률 높은 영역 찾아가는 베이지안 최적화기반 알고리즘 사용
)
study.optimize(objective, n_trials=300)  #하이퍼파라미터 시도 조합 개수

print("\n" + "="*40)
print(f"최적 Val PR-AUC : {study.best_value:.4f}")
print(f"최적 파라미터 : {study.best_params}")
print("="*40)

[I 2026-07-23 15:27:14,798] A new study created in memory with name: no-name-3fdaafd4-04a6-4419-85b9-bf9e1778e7f9
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
[I 2026-07-23 15:27:16,236] Trial 0 finished with value: 0.327642008025818 and parameters: {'C': 0.0023688639503640775, 'weight_type': 'none', 'l1_ratio': 0.15601864044243652}. Best is trial 0 with value: 0.327642008025818.
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, 


최적 Val PR-AUC : 0.3410
최적 파라미터 : {'C': 0.002353826125224433, 'weight_type': 'none', 'l1_ratio': 0.9998843828723812}


In [ ]:
# Threshold Tuning (val:2018 기준)
best_params = study.best_params

if best_params['weight_type'] == 'none':
    final_class_weight = None
elif best_params['weight_type'] == 'balanced':
    final_class_weight = 'balanced'
else:
    final_class_weight = {0: 1.0, 1: best_params['churn_weight']}

# train(2017)으로 fit → val(2018) threshold 탐색용
tuning_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', LogisticRegression(
        penalty='elasticnet',
        C=best_params['C'],
        l1_ratio=best_params['l1_ratio'],
        class_weight=final_class_weight,
        solver='saga',
        max_iter=3000,
        tol=1e-4,
        random_state=42,
        n_jobs=-1
    ))
])

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    tuning_pipeline.fit(X_train, y_train)

val_preds_proba = tuning_pipeline.predict_proba(X_val)[:, 1]

# val(2018) 기준 threshold 탐색
thresholds = np.arange(0.1, 0.9, 0.01)
best_threshold = 0.5
best_f1 = 0.0

for th in thresholds:
    preds_label = (val_preds_proba >= th).astype(int)
    f1 = f1_score(y_val, preds_label)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = th

print(f"\n최적 Threshold: {best_threshold:.2f}")
print(f"Val F1-Score  : {best_f1:.4f}")

# 최종 모델 학습 
X_train_final = data[data['period'].isin([2017, 2018])][x_cols]
y_train_final = data[data['period'].isin([2017, 2018])][target]

final_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('model', LogisticRegression(
        penalty='elasticnet',
        C=best_params['C'],
        l1_ratio=best_params['l1_ratio'],
        class_weight=final_class_weight,
        solver='saga',
        max_iter=3000,
        tol=1e-4,
        random_state=42,
        n_jobs=-1
    ))
])

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    final_pipeline.fit(X_train_final, y_train_final) #2017-2018로 최종 학습 + 전처리 파이프라인 사용하므로 원본데이터 사용

# test(2019) 예측
test_preds_proba = final_pipeline.predict_proba(X_test)[:, 1]
test_preds_05  = (test_preds_proba >= 0.5).astype(int)
test_preds_opt = (test_preds_proba >= best_threshold).astype(int)

# 성능 출력
def print_metrics(y_true, y_pred, proba, title):
    pr_auc = average_precision_score(y_true, proba)  
    auc    = roc_auc_score(y_true, proba)
    f1     = f1_score(y_true, y_pred)
    prec   = precision_score(y_true, y_pred, zero_division=0)
    rec    = recall_score(y_true, y_pred)
    cm     = confusion_matrix(y_true, y_pred)

    print(f"\n[{title}]")
    print(f" ROC-AUC : {auc:.4f} / PR-AUC: {pr_auc:.4f}")
    print(f" F1      : {f1:.4f}")
    print(f" Precision: {prec:.4f} / Recall: {rec:.4f}")
    print(f" Confusion Matrix:\n{cm}")

print("\n" + "="*60)
print("2019년 Test 최종 성능 비교")
print("="*60)
print_metrics(y_test, test_preds_05,  test_preds_proba, "Threshold=0.50")
print_metrics(y_test, test_preds_opt, test_preds_proba, f"Threshold={best_threshold:.2f}")

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



최적 Threshold: 0.14
Val F1-Score  : 0.3970


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



2019년 Test 최종 성능 비교

[Threshold=0.50]
 ROC-AUC : 0.7143 / PR-AUC: 0.3432
 F1      : 0.0748
 Precision: 0.6045 / Recall: 0.0399
 Confusion Matrix:
[[63154   318]
 [11709   486]]

[Threshold=0.14]
 ROC-AUC : 0.7143 / PR-AUC: 0.3432
 F1      : 0.3575
 Precision: 0.2314 / Recall: 0.7863
 Confusion Matrix:
[[31613 31859]
 [ 2606  9589]]


- 모델 저장 (pkl)

In [ ]:
import pickle

# 저장할 요소들을 딕셔너리로 묶기
model_pack = {
    'model': final_pipeline,           # 튜닝 완료된 로지스틱 회귀 모델
    'preprocessor': preprocessor,   # 스케일링/원핫인코딩 담당 전처리기
    'threshold': best_threshold,   
    'features': {
        'x_cols': x_cols,
        'num_cols': num_cols,
        'cat_cols': cat_cols
    },
    'best_params': study.best_params # 참고용 최적 파라미터 정보
}

# pickle 파일로 저장
file_name = 'churn_model_glm_final.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(model_pack, f)

print(f"모델 팩이 '{file_name}'으로 저장되었습니다.")

모델 팩이 'churn_model_glm_final.pkl'으로 저장되었습니다.


In [ ]:
# 저장 확인
with open(file_name, 'rb') as f:
    loaded = pickle.load(f)

preds_check = loaded['model'].predict_proba(X_test)[:, 1]
print(f"저장 전후 예측값 동일: {np.allclose(test_preds_proba, preds_check)}")

저장 전후 예측값 동일: True


In [ ]:
base_model = loaded['model']

print("🔍 [모델 팩 내부 검증 결과]")
print("=" * 50)

print(f"▶️ 모델 종류 : {type(base_model).__name__}")

print(f"\n▶️ 상세 설정값 :\n{base_model}")

print("-" * 50)

print(f"▶️ 설정된 임계값(Threshold) : {loaded.get('threshold')}")
print(f"▶️ 전처리기 종류 : {type(loaded.get('preprocessor')).__name__}")

features = loaded.get('features', {})
print(f"▶️ 학습에 사용된 전체 컬럼 수 : {len(features.get('x_cols', []))}개")
print("=" * 50)

🔍 [모델 팩 내부 검증 결과]
▶️ 모델 종류 : Pipeline

▶️ 상세 설정값 :
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['premium',
                                                   'seniority_policy',
                                                   'log_cost_claims_year',
                                                   'age', 'IICIMUN_capped',
                                                   'IICIPROV', 'C_H_num',
                                                   'C_GI', 'C_IE_T',
                                                   'missing_geo_cxt',
                                                   'high_loss',
                                                   'relative_poverty',
                                                   'kr_premium_shock',
                                                   'kr_economic_stress',
                                                   'kr_medical_de